In [13]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

# Experimentation with FlexAnomalies Federated Models

This notebook is organized into two fully separated experiment sections so each workflow has its own datasets, metrics, result tables, and timing analysis:


In [14]:
import importlib
import time

import numpy as np
import pandas as pd

from flex.pool import FlexPool
from flexanomalies.pool.aggregators_cl import aggregate_cl
from flexanomalies.pool.aggregators_favg import aggregate_ae
from flexanomalies.pool.aggregators_pca import aggregate_pca
from flexanomalies.pool.primitives_cluster import (
    build_server_model_cl,
    copy_model_to_clients_cl,
    get_clients_weights_cl,
    set_aggregated_weights_cl,
    train_cl,
)
from flexanomalies.pool.primitives_deepmodel import (
    build_server_model_ae,
    copy_model_to_clients_ae,
    set_aggregated_weights_ae,
    train_ae,
    weights_collector_ae,
)
from flexanomalies.pool.primitives_iforest import (
    aggregate_if,
    build_server_model_if,
    copy_model_to_clients_if,
    get_clients_weights_if,
    set_aggregated_weights_if,
    train_if,
)
from flexanomalies.pool.primitives_pca import (
    build_server_model_pca,
    copy_model_to_clients_pca,
    get_clients_weights_pca,
    set_aggregated_weights_pca,
    train_pca,
)
from flexanomalies.utils import (
    AutoEncoder,
    ClusterAnomaly,
    DeepCNN_LSTM,
    IsolationForest,
    PCA_Anomaly,
)
from flexanomalies.utils.load_data import federate_data

from RADAR.federated_data.algorithms import flexanomalies
import RADAR.metrics_module as metrics_module
import RADAR.static_data.anomaly_dataset_utils as anomaly_dataset_utils
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor

metrics_module = importlib.reload(metrics_module)
anomaly_dataset_utils = importlib.reload(anomaly_dataset_utils)

In [15]:
RUN_STATIC_EXPERIMENTS = True
RUN_TIME_SERIES_EXPERIMENTS = True
RUN_STATIC_TIMING = True
RUN_TIME_SERIES_TIMING = True

print('Execution flags configured:')
print(f'  RUN_STATIC_EXPERIMENTS={RUN_STATIC_EXPERIMENTS}')
print(f'  RUN_TIME_SERIES_EXPERIMENTS={RUN_TIME_SERIES_EXPERIMENTS}')
print(f'  RUN_STATIC_TIMING={RUN_STATIC_TIMING}')
print(f'  RUN_TIME_SERIES_TIMING={RUN_TIME_SERIES_TIMING}')

Execution flags configured:
  RUN_STATIC_EXPERIMENTS=True
  RUN_TIME_SERIES_EXPERIMENTS=True
  RUN_STATIC_TIMING=True
  RUN_TIME_SERIES_TIMING=True


## Experimentation with FlexAnomalies Federated Models with Static Models

This section is fully dedicated to the static-data benchmark. It reuses the same anomaly-detection framing as `experiment_pyod_models.ipynb`, and keeps its own datasets, result table, exported files, and timing analysis.

In [16]:
static_dataset_configs = {}
static_summary_df = pd.DataFrame()

if RUN_STATIC_EXPERIMENTS or RUN_STATIC_TIMING:
    static_dataset_configs = {
        'shuttle': anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
            dataset_name='shuttle',
            normal_label=1,
            target_test_contamination=0.1,
            max_train_normals=8000,
            max_test_size=5000,
        ),
        'arrhythmia': anomaly_dataset_utils.build_loaded_uci_anomaly_dataset(
            dataset_name='arrhythmia',
            normal_label=1,
            target_test_contamination=0.1,
        ),
    }

    static_summary_df = pd.DataFrame([
        {
            'dataset': dataset_name,
            'samples': config['n_samples'],
            'features': config['n_features'],
            'original_anomaly_ratio': round(config['original_positive_ratio'], 4),
            'benchmark_test_contamination': round(config['benchmark_test_positive_ratio'], 4),
            'train_normals_used': config['train_normals'],
            'test_normals': config['test_normals'],
            'test_anomalies': config['test_anomalies'],
        }
        for dataset_name, config in static_dataset_configs.items()
    ]).reset_index(drop=True)

    display(static_summary_df)
else:
    print('Static section disabled. Enable RUN_STATIC_EXPERIMENTS or RUN_STATIC_TIMING to prepare these datasets.')

,dataset,samples,features,original_anomaly_ratio,benchmark_test_contamination,train_normals_used,test_normals,test_anomalies
0,shuttle,58000,7,0.214,0.1036,8000,4482,518
1,arrhythmia,452,279,0.458,0.0926,196,49,5


In [ ]:
def flatten_1d(values):
    return np.asarray(values).astype(float).ravel()


def binary_1d(values):
    return np.asarray(values).astype(int).ravel()


def compute_metrics_row(y_true, y_pred, scores):
    y_true = binary_1d(y_true)
    y_pred = binary_1d(y_pred)
    scores = flatten_1d(scores)

    limit = min(len(y_true), len(y_pred), len(scores))
    y_true = y_true[:limit]
    y_pred = y_pred[:limit]
    scores = scores[:limit]

    finite_scores = np.isfinite(scores)
    if finite_scores.all() and len(np.unique(y_true)) > 1:
        roc_auc = metrics_module.metric_AUC_ROC_scores(y_true, scores)
        pr_auc = metrics_module.metric_PR_AUC(y_true, scores)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    return {
        'accuracy': round(metrics_module.metric_accuracy(y_true, y_pred) / 100, 4),
        'precision': round(metrics_module.metric_precision(y_true, y_pred), 4),
        'recall': round(metrics_module.metric_recall(y_true, y_pred), 4),
        'f1': round(metrics_module.metric_F1score(y_true, y_pred), 4),
        'roc_auc_scores': round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
        'pr_auc_scores': round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
        'evaluated_samples': int(limit),
    }


def summarize_mse(scores):
    scores = np.asarray(scores, dtype=float).ravel()
    return float(scores.mean()) if scores.size else np.nan


direct_flex_model_classes = {
    'isolationForest': IsolationForest,
    'pcaAnomaly': PCA_Anomaly,
    'clusterAnomaly': ClusterAnomaly,
    'autoencoder': AutoEncoder,
    'deepCNN_LSTM': DeepCNN_LSTM,
}


direct_federated_ops = {
    'isolationForest': {
        'build_model': build_server_model_if,
        'copy': copy_model_to_clients_if,
        'train': train_if,
        'collect': get_clients_weights_if,
        'aggregate': aggregate_if,
        'set_weights': set_aggregated_weights_if,
    },
    'pcaAnomaly': {
        'build_model': build_server_model_pca,
        'copy': copy_model_to_clients_pca,
        'train': train_pca,
        'collect': get_clients_weights_pca,
        'aggregate': aggregate_pca,
        'set_weights': set_aggregated_weights_pca,
    },
    'clusterAnomaly': {
        'build_model': build_server_model_cl,
        'copy': copy_model_to_clients_cl,
        'train': train_cl,
        'collect': get_clients_weights_cl,
        'aggregate': aggregate_cl,
        'set_weights': set_aggregated_weights_cl,
    },
    'autoencoder': {
        'build_model': build_server_model_ae,
        'copy': copy_model_to_clients_ae,
        'train': train_ae,
        'collect': weights_collector_ae,
        'aggregate': aggregate_ae,
        'set_weights': set_aggregated_weights_ae,
    },
    'deepCNN_LSTM': {
        'build_model': build_server_model_ae,
        'copy': copy_model_to_clients_ae,
        'train': train_ae,
        'collect': weights_collector_ae,
        'aggregate': aggregate_ae,
        'set_weights': set_aggregated_weights_ae,
    },
}


def extract_prediction_labels(model_object, prediction_output):
    for candidate in (
        getattr(model_object, 'labels_', None),
        getattr(getattr(model_object, 'model', None), 'labels_', None),
        getattr(prediction_output, 'labels_', None),
    ):
        if candidate is not None:
            return binary_1d(candidate)
    return binary_1d(prediction_output)


def predict_and_score_model(model_object, X, y=None):
    prediction_output = model_object.predict(X, y) if y is not None else model_object.predict(X)
    labels = extract_prediction_labels(model_object, prediction_output)
    scores = model_object.decision_function(X, y) if y is not None else model_object.decision_function(X)
    return labels, flatten_1d(scores)


def build_direct_model_kwargs(model_kwargs):
    excluded_keys = {'algorithm_', 'label_parser', 'n_clients', 'n_rounds'}
    return {
        key: value
        for key, value in model_kwargs.items()
        if key not in excluded_keys
    }


def train_platform_model(model_kwargs, X_train, y_train):
    model = flexanomalies.FlexAnomalyDetection(**model_kwargs)
    model.fit(X_train, y_train)
    return model


def train_direct_federated_model(model_kwargs, X_train, y_train):
    algorithm_name = model_kwargs['algorithm_']
    direct_model_cls = direct_flex_model_classes[algorithm_name]
    direct_model = direct_model_cls(**build_direct_model_kwargs(model_kwargs))
    federated_ops = direct_federated_ops[algorithm_name]

    flex_dataset = federate_data(model_kwargs['n_clients'], X_train, y_train)
    pool = FlexPool.client_server_pool(
        fed_dataset=flex_dataset,
        server_id=f'{algorithm_name}_server',
        init_func=federated_ops['build_model'],
        model=direct_model,
    )

    for _ in range(model_kwargs['n_rounds']):
        pool.servers.map(federated_ops['copy'], pool.clients)
        pool.clients.map(federated_ops['train'])
        pool.aggregators.map(federated_ops['collect'], pool.clients)
        if algorithm_name == 'clusterAnomaly':
            pool.aggregators.map(federated_ops['aggregate'], model=direct_model)
        else:
            pool.aggregators.map(federated_ops['aggregate'])
        pool.aggregators.map(federated_ops['set_weights'], pool.servers)

    return pool.servers._models[f'{algorithm_name}_server']['model']


def measure_training_time(train_callable):
    start_time = time.perf_counter()
    trained_model = train_callable()
    return time.perf_counter() - start_time, trained_model


static_model_configs = [
    {
        'algorithm_': 'isolationForest',
        'n_estimators': 100,
        'n_clients': 5,
        'n_rounds': 5,
    },
    {
        'algorithm_': 'pcaAnomaly',
        'preprocess': False,
        'n_components': 4,
        'n_clients': 5,
        'n_rounds': 5,
    },
    {
        'algorithm_': 'clusterAnomaly',
        'n_clusters': 4,
        'n_clients': 5,
        'n_rounds': 5,
    },
]

static_results = []
static_timing_results = []

if RUN_STATIC_EXPERIMENTS or RUN_STATIC_TIMING:
    for dataset_name, config in static_dataset_configs.items():
        print(f'\nStatic dataset: {dataset_name}')
        y_train_dummy = np.zeros(config['X_train'].shape[0], dtype=int)

        for model_params in static_model_configs:
            model_kwargs = {
                **model_params,
                'contamination': float(config['benchmark_test_positive_ratio']),
                'label_parser': None,
            }

            platform_time_s, platform_model = measure_training_time(
                lambda mk=model_kwargs, ds=config, yt=y_train_dummy: train_platform_model(mk, ds['X_train'], yt)
            )
            platform_predictions, platform_scores = predict_and_score_model(platform_model, config['X_test'])
            platform_metrics = compute_metrics_row(config['y_test'], platform_predictions, platform_scores)

            if RUN_STATIC_EXPERIMENTS:
                result_row = {
                    'data_type': 'static',
                    'dataset': dataset_name,
                    'algorithm': model_params['algorithm_'],
                    'contamination': round(float(config['benchmark_test_positive_ratio']), 4),
                    **platform_metrics,
                }
                static_results.append(result_row)
                print(result_row)

            if RUN_STATIC_TIMING:
                base_time_s, base_model = measure_training_time(
                    lambda mk=model_kwargs, ds=config, yt=y_train_dummy: train_direct_federated_model(mk, ds['X_train'], yt)
                )
                base_predictions, base_scores = predict_and_score_model(base_model, config['X_test'])
                base_metrics = compute_metrics_row(config['y_test'], base_predictions, base_scores)

                static_timing_results.append({
                    'dataset': dataset_name,
                    'algorithm': model_params['algorithm_'],
                    'timing_repetitions': 1,
                    'platform_time_s': round(platform_time_s, 4),
                    'base_time_s': round(base_time_s, 4),
                    'overhead_s': round(platform_time_s - base_time_s, 4),
                    'speedup_base_over_platform': round(base_time_s / platform_time_s, 4) if platform_time_s > 0 else np.nan,
                    'platform_roc_auc_scores': platform_metrics['roc_auc_scores'],
                    'base_roc_auc_scores': base_metrics['roc_auc_scores'],
                    'roc_auc_diff': round(platform_metrics['roc_auc_scores'] - base_metrics['roc_auc_scores'], 4)
                    if np.isfinite(platform_metrics['roc_auc_scores']) and np.isfinite(base_metrics['roc_auc_scores'])
                    else np.nan,
                    'platform_pr_auc_scores': platform_metrics['pr_auc_scores'],
                    'base_pr_auc_scores': base_metrics['pr_auc_scores'],
                    'pr_auc_diff': round(platform_metrics['pr_auc_scores'] - base_metrics['pr_auc_scores'], 4)
                    if np.isfinite(platform_metrics['pr_auc_scores']) and np.isfinite(base_metrics['pr_auc_scores'])
                    else np.nan,
                })
else:
    print('Static experiment section skipped.')

static_results_df = pd.DataFrame(static_results)
if not static_results_df.empty:
    static_results_df = static_results_df.sort_values(
        ['dataset', 'pr_auc_scores', 'roc_auc_scores'],
        ascending=[True, False, False],
        na_position='last',
    ).reset_index(drop=True)
else:
    print('No static results available.')

### Static Models Results

This subsection stores and displays only the results produced by the static federated models.

In [ ]:
static_results_path = results_dir / 'uci_flexanomalies_static_results.csv'

if not static_results_df.empty:
    static_results_df.to_csv(static_results_path, index=False)
    print(f'Saved static results to: {static_results_path}')
    display(static_results_df)
else:
    print('Static results were not saved because the DataFrame is empty.')

### Static Models Timing

This subsection summarizes the training times captured during the same static-model executions used to generate the results table, so the models are not retrained here.

In [19]:
STATIC_TIMING_REPETITIONS = 1

static_timing_results_df = pd.DataFrame(static_timing_results)
if RUN_STATIC_TIMING and not static_timing_results_df.empty:
    static_timing_results_df = static_timing_results_df.sort_values(
        ['dataset', 'speedup_base_over_platform'],
        ascending=[True, False],
        na_position='last',
    ).reset_index(drop=True)

    display(static_timing_results_df)
elif RUN_STATIC_TIMING:
    print('No static timing results available.')
else:
    print('Static timing section skipped.')

,dataset,algorithm,timing_repetitions,platform_time_s,base_time_s,overhead_s,speedup_base_over_platform,platform_roc_auc_scores,base_roc_auc_scores,roc_auc_diff,platform_pr_auc_scores,base_pr_auc_scores,pr_auc_diff
0,arrhythmia,clusterAnomaly,1,0.1640,0.1508,0.0132,0.9194,0.7510,0.7510,0.0000,0.5288,0.5267,0.0021
1,arrhythmia,isolationForest,1,4.2695,3.5734,0.6960,0.8370,0.7020,0.7020,0.0000,0.1988,0.2074,-0.0086
2,arrhythmia,pcaAnomaly,1,0.4315,0.2424,0.1891,0.5619,0.7673,0.7673,0.0000,0.5309,0.5309,0.0000
3,shuttle,pcaAnomaly,1,0.0818,0.1182,-0.0364,1.4455,0.7974,0.7974,0.0000,0.5860,0.5860,0.0000
4,shuttle,isolationForest,1,3.8827,4.0236,-0.1409,1.0363,0.9234,0.8959,0.0275,0.7108,0.6441,0.0667
5,shuttle,clusterAnomaly,1,0.1592,0.1206,0.0385,0.7578,0.9695,0.9656,0.0039,0.7965,0.7956,0.0009


In [20]:
static_timing_results_path = results_dir / 'uci_flexanomalies_static_timing_results.csv'

if not static_timing_results_df.empty:
    static_timing_results_df.to_csv(static_timing_results_path, index=False)
    print(f'Saved static timing results to: {static_timing_results_path}')
else:
    print('Static timing results were not saved because the DataFrame is empty.')

Saved static timing results to: /data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/results/uci_flexanomalies_static_timing_results.csv


## Experimentation with FlexAnomalies Federated Models with Time-Series Models

This section is fully dedicated to the time-series benchmark. It reuses the same raw datasets and preprocessing ideas as `experiment_transformers_models.ipynb`, and keeps its own datasets, result table, exported files, and timing analysis. The reported quality metric in this section is reconstruction `MSE`.

In [ ]:
WINDOW_SIZE = 24
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.reshape(y_windows.shape[0], -1).sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)
    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train': np.asarray(X_train, dtype=np.float32),
        'X_test': np.asarray(X_test, dtype=np.float32),
        'y_train': np.asarray(y_train, dtype=int),
        'y_test': np.asarray(y_test, dtype=int),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)
    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train': np.asarray(X_train, dtype=np.float32),
        'X_test': np.asarray(X_test, dtype=np.float32),
        'y_train': np.asarray(y_train, dtype=int),
        'y_test': np.asarray(y_test, dtype=int),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

time_series_dataset_configs = {}
time_series_summary_df = pd.DataFrame()

if RUN_TIME_SERIES_EXPERIMENTS or RUN_TIME_SERIES_TIMING:
    time_series_dataset_configs = {
        'ai4i': prepare_ai4i_dataset(),
        'metro_interstate': prepare_metro_dataset(),
    }

    time_series_summary_df = pd.DataFrame([
        {
            'dataset_key': dataset_key,
            'dataset_name': config['dataset'],
            'samples': config['n_samples'],
            'features': config['n_features'],
            'positive_ratio_points': config['positive_ratio_points'],
            'label_note': config['label_note'],
        }
        for dataset_key, config in time_series_dataset_configs.items()
    ]).reset_index(drop=True)

    display(time_series_summary_df)
else:
    print('Time-series section disabled. Enable RUN_TIME_SERIES_EXPERIMENTS or RUN_TIME_SERIES_TIMING to prepare these datasets.')

In [ ]:
def align_binary_targets(targets, expected_len):
    targets = np.asarray(targets)
    window_targets = aggregate_window_labels(targets)
    flat_targets = binary_1d(targets)

    if len(window_targets) == expected_len:
        return window_targets
    if len(flat_targets) == expected_len:
        return flat_targets
    if len(window_targets) > expected_len:
        return window_targets[:expected_len]
    if len(flat_targets) > expected_len:
        return flat_targets[:expected_len]
    return np.resize(window_targets, expected_len).astype(int)


def align_scores(scores, expected_len):
    scores = flatten_1d(scores)
    if len(scores) >= expected_len:
        return scores[:expected_len]
    return np.resize(scores, expected_len)


def resolve_contamination_ratio(ratio, default=0.1, min_value=0.01, max_value=0.5):
    if ratio is None or not np.isfinite(ratio):
        return float(default)
    return float(np.clip(ratio, min_value, max_value - 1e-6))


def build_autoencoder_windows(config, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(
        config['X_train'], config['y_train'], config['X_test'], config['y_test']
    )
    return X_train_windows, y_train_windows, X_test_windows, y_test_windows, aggregate_window_labels(y_test_windows)


def build_forecasting_windows(config, window_size=WINDOW_SIZE, step_size=STEP_SIZE, n_pred=1):
    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=True, n_pred=n_pred)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows, label_test_windows = processor.process_train_test(
        config['X_train'], config['y_train'], config['X_test'], config['y_test'], l_test=config['y_test']
    )
    return X_train_windows, y_train_windows, X_test_windows, y_test_windows, aggregate_window_labels(label_test_windows)


def compute_time_series_metrics_row(y_true, y_pred, scores):
    y_true = binary_1d(y_true)
    y_pred = binary_1d(y_pred)
    scores = flatten_1d(scores)

    limit = min(len(y_true), len(y_pred), len(scores))
    y_true = y_true[:limit]
    y_pred = y_pred[:limit]
    scores = scores[:limit]

    finite_scores = bool(np.isfinite(scores).all())
    mse = summarize_mse(scores) if finite_scores else np.nan

    return {
        'accuracy': round(metrics_module.metric_accuracy(y_true, y_pred) / 100, 4),
        'precision': round(metrics_module.metric_precision(y_true, y_pred), 4),
        'recall': round(metrics_module.metric_recall(y_true, y_pred), 4),
        'mse': round(float(mse), 6) if np.isfinite(mse) else np.nan,
        'evaluated_samples': int(limit),
    }


time_series_model_configs = [
    {
        'algorithm_': 'autoencoder',
        'builder': build_autoencoder_windows,
        'base_kwargs': {
            'epochs': 5,
            'batch_size': 16,
            'neurons': [32, 16, 32],
            'hidden_act': ['relu', 'relu', 'relu'],
            'preprocess': False,
            'w_size': WINDOW_SIZE,
            'n_pred': 1,
            'n_clients': 3,
            'n_rounds': 5,
        },
    },
    {
        'algorithm_': 'deepCNN_LSTM',
        'builder': build_forecasting_windows,
        'base_kwargs': {
            'epochs': 5,
            'batch_size': 8,
            'filters_cnn': [8, 6],
            'units_lstm': [8, 6],
            'kernel_size': [4, 4],
            'hidden_act': ['relu', 'relu'],
            'w_size': WINDOW_SIZE,
            'n_pred': 1,
            'n_clients': 3,
            'n_rounds': 3,
        },
    },
]

time_series_results = []
time_series_timing_results = []

if RUN_TIME_SERIES_EXPERIMENTS or RUN_TIME_SERIES_TIMING:
    for dataset_key, config in time_series_dataset_configs.items():
        print(f'\nTime-series dataset: {config["dataset"]}')

        for model_config in time_series_model_configs:
            X_train_windows, y_train_windows, X_test_windows, y_test_windows, y_eval_reference = model_config['builder'](config)
            contamination = resolve_contamination_ratio(config.get('positive_ratio_points'))

            model_kwargs = {
                'algorithm_': model_config['algorithm_'],
                'contamination': contamination,
                'label_parser': None,
                'input_dim': int(config['n_features']),
                **model_config['base_kwargs'],
            }

            platform_time_s, platform_model = measure_training_time(
                lambda mk=model_kwargs, Xw=X_train_windows, yw=y_train_windows: train_platform_model(mk, Xw, yw)
            )

            if model_config['algorithm_'] == 'deepCNN_LSTM':
                platform_predictions, platform_raw_scores = predict_and_score_model(platform_model, X_test_windows, y_test_windows)
            else:
                platform_predictions, platform_raw_scores = predict_and_score_model(platform_model, X_test_windows)

            platform_scores = align_scores(platform_raw_scores, len(platform_predictions))
            y_true_platform = align_binary_targets(y_eval_reference, len(platform_predictions))
            platform_metrics = compute_time_series_metrics_row(y_true_platform, platform_predictions, platform_scores)

            if RUN_TIME_SERIES_EXPERIMENTS:
                result_row = {
                    'data_type': 'time_series',
                    'dataset': dataset_key,
                    'dataset_name': config['dataset'],
                    'algorithm': model_config['algorithm_'],
                    'window_size': WINDOW_SIZE,
                    'contamination': round(contamination, 4),
                    'train_windows': int(len(X_train_windows)),
                    'test_windows': int(len(X_test_windows)),
                    **platform_metrics,
                }
                time_series_results.append(result_row)
                print(result_row)

            if RUN_TIME_SERIES_TIMING:
                base_time_s, base_model = measure_training_time(
                    lambda mk=model_kwargs, Xw=X_train_windows, yw=y_train_windows: train_direct_federated_model(mk, Xw, yw)
                )

                if model_config['algorithm_'] == 'deepCNN_LSTM':
                    base_predictions, base_raw_scores = predict_and_score_model(base_model, X_test_windows, y_test_windows)
                else:
                    base_predictions, base_raw_scores = predict_and_score_model(base_model, X_test_windows)

                base_scores = align_scores(base_raw_scores, len(base_predictions))
                y_true_base = align_binary_targets(y_eval_reference, len(base_predictions))
                base_metrics = compute_time_series_metrics_row(y_true_base, base_predictions, base_scores)

                time_series_timing_results.append({
                    'dataset': dataset_key,
                    'dataset_name': config['dataset'],
                    'algorithm': model_config['algorithm_'],
                    'timing_repetitions': 1,
                    'platform_time_s': round(platform_time_s, 4),
                    'base_time_s': round(base_time_s, 4),
                    'overhead_s': round(platform_time_s - base_time_s, 4),
                    'speedup_base_over_platform': round(base_time_s / platform_time_s, 4) if platform_time_s > 0 else np.nan,
                    'platform_mse': platform_metrics['mse'],
                    'base_mse': base_metrics['mse'],
                    'mse_diff': round(platform_metrics['mse'] - base_metrics['mse'], 6)
                    if np.isfinite(platform_metrics['mse']) and np.isfinite(base_metrics['mse'])
                    else np.nan,
                })
else:
    print('Time-series experiment section skipped.')

time_series_results_df = pd.DataFrame(time_series_results)
if not time_series_results_df.empty:
    time_series_results_df = pd.DataFrame(time_series_results).sort_values(
        ['dataset', 'mse'],
        ascending=[True, True],
        na_position='last',
    ).reset_index(drop=True)
else:
    print('No time-series results available.')

In [ ]:
time_series_results_path = results_dir / 'uci_flexanomalies_time_series_results.csv'

if not time_series_results_df.empty:
    time_series_results_df.to_csv(time_series_results_path, index=False)
    print(f'Saved time-series results to: {time_series_results_path}')
    display(time_series_results_df)
else:
    print('Time-series results were not saved because the DataFrame is empty.')

### Time-Series Models Timing

This subsection summarizes the training times captured during the same time-series executions used to generate the results table, so the models are not retrained here.

- `platform_time_s`: average training time using `RADAR.federated_data.algorithms.flexanomalies.FlexAnomalyDetection`.
- `base_time_s`: average training time using the direct `flexanomalies` library flow.
- `overhead_s`: difference `platform_time_s - base_time_s`.
- `speedup_base_over_platform`: ratio `base_time_s / platform_time_s`.
- `platform_mse`, `base_mse`, and `mse_diff`: reconstruction error comparison for the RADAR and direct execution paths.

In [ ]:
TIME_SERIES_TIMING_REPETITIONS = 1

time_series_timing_results_df = pd.DataFrame(time_series_timing_results)
if RUN_TIME_SERIES_TIMING and not time_series_timing_results_df.empty:
    time_series_timing_results_df = time_series_timing_results_df.sort_values(
        ['dataset', 'speedup_base_over_platform'],
        ascending=[True, False],
        na_position='last',
    ).reset_index(drop=True)

    display(time_series_timing_results_df)
elif RUN_TIME_SERIES_TIMING:
    print('No time-series timing results available.')
else:
    print('Time-series timing section skipped.')

In [ ]:
time_series_timing_results_path = results_dir / 'uci_flexanomalies_time_series_timing_results.csv'

if not time_series_timing_results_df.empty:
    time_series_timing_results_df.to_csv(time_series_timing_results_path, index=False)
    print(f'Saved time-series timing results to: {time_series_timing_results_path}')
else:
    print('Time-series timing results were not saved because the DataFrame is empty.')